# Modul 1: Tuning kontrolera PD — wplyw Kp i Kd na zachowanie robota

W tym notebooku zbadamy, jak parametry kontrolera PD wplywaja na zachowanie robota G1.

Dowiesz sie:
1. Jak **Kp** (wzmocnienie proporcjonalne) wplywa na sztywnosc stawow
2. Jak **Kd** (wzmocnienie rozniczkujace) tlumi oscylacje
3. Jakie minimalne wartosci sa potrzebne, zeby robot stal

---
*Kurs: AI & Reinforcement Learning dla robotyki*

In [ ]:
# Instalacja wymaganych pakietow
!pip install -q mujoco mujoco_menagerie numpy matplotlib

In [ ]:
import mujoco
import mujoco_menagerie
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Zaladuj model G1
model_path = Path(mujoco_menagerie.__path__[0]) / "unitree_g1" / "scene.xml"
print(f"Model: {model_path}")
print(f"MuJoCo: {mujoco.__version__}")

## Teoria kontrolera PD

Kontroler PD oblicza moment sily (torque) na kazdym stawie wedlug wzoru:

$$\tau = K_p \cdot (q_{target} - q) + K_d \cdot (0 - \dot{q})$$

Gdzie:
- $\tau$ — moment sily na stawie [Nm]
- $K_p$ — wzmocnienie **proporcjonalne** ("sprezyna")
- $K_d$ — wzmocnienie **rozniczkujace** ("amortyzator")
- $q_{target}$ — docelowa pozycja stawu (z keyframe "stand")
- $q$ — aktualna pozycja stawu
- $\dot{q}$ — aktualna predkosc stawu

**Intuicja:**
- **Kp za niskie** — robot jest "miekki", nie utrzymuje pozycji
- **Kp za wysokie** — robot jest sztywny, ale moze oscylowac
- **Kd za niskie** — brak tlumienia, robot "drzy"
- **Kd za wysokie** — robot jest powolny, reaguje ospale

In [ ]:
def simulate_pd(kp, kd, duration=3.0):
    """
    Symuluje robota G1 z kontrolerem PD o zadanych wzmocnieniach.

    Args:
        kp: wzmocnienie proporcjonalne
        kd: wzmocnienie rozniczkujace
        duration: czas symulacji [s]

    Returns:
        times: lista chwil czasowych
        heights: lista wysokosci robota
        final_img: obraz koncowej klatki (numpy array)
    """
    model = mujoco.MjModel.from_xml_path(str(model_path))
    data = mujoco.MjData(model)

    # Reset do keyframe "stand"
    if model.nkey > 0:
        kf_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_KEY, 0)
        mujoco.mj_resetData(model, data)
        data.qpos[:] = model.keyframe(kf_name).qpos
        data.qvel[:] = 0
    mujoco.mj_forward(model, data)

    target_qpos = data.qpos[7:].copy()
    dt = model.opt.timestep
    steps = int(duration / dt)

    times = []
    heights = []

    for step in range(steps):
        pos_error = target_qpos - data.qpos[7:]
        vel = data.qvel[6:]
        torques = kp * pos_error + kd * (0.0 - vel)
        torques = np.clip(torques, -200, 200)
        data.ctrl[:] = torques

        mujoco.mj_step(model, data)

        times.append(data.time)
        heights.append(data.qpos[2])

    # Renderuj koncowa klatke
    renderer = mujoco.Renderer(model, height=360, width=480)
    renderer.update_scene(data, camera="track")
    final_img = renderer.render().copy()

    return times, heights, final_img

## Porownanie roznych Kp

Sprawdzmy, jak rozne wartosci **Kp** wplywaja na wysokosc robota.

Kd jest stale = 5. Testujemy Kp od bardzo niskiego (10) do bardzo wysokiego (2000).

In [ ]:
# Porownanie roznych Kp (Kd=5 stale)
kp_values = [10, 50, 200, 500, 2000]
kd_fixed = 5.0

plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(kp_values)))

results_kp = {}
for kp, color in zip(kp_values, colors):
    print(f"Symulacja: Kp={kp}, Kd={kd_fixed} ...", end=" ")
    times, heights, img = simulate_pd(kp, kd_fixed)
    results_kp[kp] = (times, heights, img)
    plt.plot(times, heights, linewidth=2, color=color, label=f'Kp={kp}')
    print(f"koncowa wysokosc: {heights[-1]:.3f}m")

plt.xlabel('Czas [s]', fontsize=12)
plt.ylabel('Wysokosc [m]', fontsize=12)
plt.title(f'Wplyw Kp na stabilnosc robota (Kd={kd_fixed})', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axhline(y=0.5, color='red', linestyle=':', alpha=0.5, label='Prog upadku')
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

## Porownanie roznych Kd

Teraz sprawdzmy wplyw **Kd** (tlumienia) przy stalym Kp = 200.

In [ ]:
# Porownanie roznych Kd (Kp=200 stale)
kp_fixed = 200.0
kd_values = [0, 1, 5, 20, 50]

plt.figure(figsize=(12, 6))
colors = plt.cm.plasma(np.linspace(0, 1, len(kd_values)))

results_kd = {}
for kd, color in zip(kd_values, colors):
    print(f"Symulacja: Kp={kp_fixed}, Kd={kd} ...", end=" ")
    times, heights, img = simulate_pd(kp_fixed, kd)
    results_kd[kd] = (times, heights, img)
    plt.plot(times, heights, linewidth=2, color=color, label=f'Kd={kd}')
    print(f"koncowa wysokosc: {heights[-1]:.3f}m")

plt.xlabel('Czas [s]', fontsize=12)
plt.ylabel('Wysokosc [m]', fontsize=12)
plt.title(f'Wplyw Kd na stabilnosc robota (Kp={kp_fixed})', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axhline(y=0.5, color='red', linestyle=':', alpha=0.5, label='Prog upadku')
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

## Wizualizacja — niskie vs wysokie Kp

Porownajmy wizualnie, jak wyglada robot z niskim Kp ("miekki") vs wysokim Kp ("sztywny") po 2 sekundach symulacji.

In [ ]:
# Wizualne porownanie: niskie vs wysokie Kp
kp_low = 10
kp_high = 500
kd_viz = 5.0

# Symulacja z renderowaniem w t=2s
def simulate_and_render_at(kp, kd, t_render=2.0):
    model = mujoco.MjModel.from_xml_path(str(model_path))
    data = mujoco.MjData(model)

    if model.nkey > 0:
        kf_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_KEY, 0)
        mujoco.mj_resetData(model, data)
        data.qpos[:] = model.keyframe(kf_name).qpos
        data.qvel[:] = 0
    mujoco.mj_forward(model, data)

    target_qpos = data.qpos[7:].copy()
    dt = model.opt.timestep
    steps = int(t_render / dt)

    for step in range(steps):
        pos_error = target_qpos - data.qpos[7:]
        vel = data.qvel[6:]
        torques = kp * pos_error + kd * (0.0 - vel)
        torques = np.clip(torques, -200, 200)
        data.ctrl[:] = torques
        mujoco.mj_step(model, data)

    renderer = mujoco.Renderer(model, height=480, width=640)
    renderer.update_scene(data, camera="track")
    return renderer.render().copy(), data.qpos[2]

img_low, h_low = simulate_and_render_at(kp_low, kd_viz, t_render=2.0)
img_high, h_high = simulate_and_render_at(kp_high, kd_viz, t_render=2.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img_low)
axes[0].set_title(f'Kp={kp_low}, Kd={kd_viz}\nWysokosc: {h_low:.3f}m', fontsize=13, color='red' if h_low < 0.5 else 'green')
axes[0].axis('off')

axes[1].imshow(img_high)
axes[1].set_title(f'Kp={kp_high}, Kd={kd_viz}\nWysokosc: {h_high:.3f}m', fontsize=13, color='red' if h_high < 0.5 else 'green')
axes[1].axis('off')

plt.suptitle('Porownanie: niskie vs wysokie Kp (t=2s)', fontsize=15)
plt.tight_layout()
plt.show()

print(f"Kp={kp_low}: robot {'stoi' if h_low > 0.5 else 'upadl/osiada'} (h={h_low:.3f}m)")
print(f"Kp={kp_high}: robot {'stoi' if h_high > 0.5 else 'upadl/osiada'} (h={h_high:.3f}m)")

## Wnioski

### Co robi Kp (wzmocnienie proporcjonalne)?
- **Niskie Kp** (np. 10) — robot jest "miekki", stawy nie generuja wystarczajacych momentow zeby przeciwdzialac grawitacji. Robot osiada lub pada.
- **Srednie Kp** (np. 200) — robot stoi stabilnie. Stawy sa wystarczajaco sztywne.
- **Wysokie Kp** (np. 2000) — robot jest bardzo sztywny, ale moze oscylowac jesli Kd jest za niskie.

### Co robi Kd (wzmocnienie rozniczkujace)?
- **Kd = 0** — brak tlumienia. Robot moze oscylowac lub byc niestabilny.
- **Niskie Kd** (np. 1) — minimalne tlumienie. Oscylacje gasna powoli.
- **Srednie Kd** (np. 5-20) — dobre tlumienie. Robot szybko stabilizuje sie.
- **Wysokie Kd** (np. 50) — robot jest "lepki", reaguje powoli na zmiany.

### Minimalne wartosci dla stania
- Dla calego ciala: **Kp >= 150**, **Kd >= 5**
- Bezpieczny start dla eksperymentow: **Kp=200, Kd=10**

### Dlaczego to wazne dla RL?
W Reinforcement Learning siec neuronowa **uczy sie** optymalnych wartosci Kp i Kd (lub bezposrednio momentow) dla kazdego stawu osobno, w zaleznosci od sytuacji. To pozwala na bardziej zaawansowane zachowania niz staly kontroler PD.

## Cwiczenie

Zmien wartosci Kp i Kd ponizej i uruchom komorke, zeby zobaczyc efekt!

**Pytania do zbadania:**
1. Jaka jest minimalna wartosc Kp, przy ktorej robot jeszcze stoi (Kd=10)?
2. Co sie stanie, gdy ustawisz Kd=0 a Kp=500? Czy robot oscyluje?
3. Czy istnieje kombinacja Kp/Kd, przy ktorej robot stoi, ale sie "trzesie"?
4. Jakie Kp/Kd sa najlepsze dla stabilnego stania?

In [ ]:
# === ZMIEN TE WARTOSCI I URUCHOM KOMORKE ===
moje_kp = 200.0   # Zmien!
moje_kd = 10.0     # Zmien!
# ============================================

print(f"Testowanie: Kp={moje_kp}, Kd={moje_kd}")
print("="*40)

times, heights, img = simulate_pd(moje_kp, moje_kd, duration=3.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wykres wysokosci
axes[0].plot(times, heights, 'b-', linewidth=2)
axes[0].set_xlabel('Czas [s]')
axes[0].set_ylabel('Wysokosc [m]')
axes[0].set_title(f'Wysokosc robota (Kp={moje_kp}, Kd={moje_kd})')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0.5, color='red', linestyle=':', alpha=0.5)
axes[0].set_ylim(bottom=0)

# Obraz koncowy
axes[1].imshow(img)
axes[1].set_title(f'Koncowa poza (t=3s)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

final_h = heights[-1]
print(f"\nKoncowa wysokosc: {final_h:.3f}m")
if final_h > 0.7:
    print("Robot STOI stabilnie!")
elif final_h > 0.4:
    print("Robot OSIADA — Kp moze byc za niskie.")
else:
    print("Robot UPADL — sprobuj wyzsze Kp lub Kd.")